# A2.2 · The bootstrap problem

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Proving identity before you hold a credential — usually solved with a long-lived secret in a file.

**Control.** Workload attestation: SPIFFE SVIDs, instance identity documents, projected SA tokens, re-attestation on rotation.

**This lab.** Issue a workload identity with zero static secrets.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE, kind |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.2"))

The bootstrap problem: an agent needs an identity before it can prove anything, and whatever you use to hand it that first credential is your real trust root.

In [ ]:
from cybercommons import identity, sandbox

# The anti-pattern: a long-lived secret in the environment. Anything that can
# read the process environment is now the agent.
print("bootstrap A — static secret in env")
print("  lifetime: until someone rotates it (median: never)")
print("  theft:    any file read, any log line, any core dump")
guard = sandbox.PathGuard(workspace="/work")
print("  ", guard.check("/work/.env"))

print("\nbootstrap B — short-lived, attested, narrowed at issue")
t = identity.mint("alice", {"repo:read", "repo:write"})
agent = identity.exchange(t, "patch-agent", {"repo:write"})
print(f"  ttl {agent.ttl:.0f}s, scopes {sorted(agent.scopes)}, chain {agent.chain()}")
print("  theft window is the ttl, and the stolen token names its own thief")

SPIFFE/SPIRE solves this properly by attesting the *workload* — the node and process identity become the evidence, so no secret has to be planted anywhere. The modelled version above captures the property that matters: the credential is short-lived and already narrowed when it arrives.

### Expect

The `.env` read is denied by the path guard, and the delegated token prints a 300-second TTL with a single narrowed scope and a readable chain.

### Your turn

For one agent you run, measure the actual lifetime of its bootstrap credential. If the answer is 'since we deployed it', that is a standing grant, not a bootstrap.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*